# Inflow & MOFA-Flex: Spatially-Resolved Cell-Cell Communication Programs

## Overview

This tutorial demonstrates how to integrate **inflow score** with **MOFA-Flex** to extract spatially-resolved, single-cell derived communication programs from mouse brain tissue. 

For a detailed walkthrough of the inflow score itself, please see our tutorial: [Inferring Cell–Cell Interactions at Single Cell Resolution](https://github.com/saezlab/liana-py/blob/main/docs/notebooks/inflow_score.ipynb).

For more information about [MOFA-Flex](https://www.biorxiv.org/content/10.1101/2025.11.03.686250v1), please visit their [documentation](https://mofaflex.readthedocs.io/stable/).

<img src="Inflow_Mofaflex_overview.png" width=1000 />

Image made with BioRender.com

---

## Load Packages

> **Note:** This tutorial uses the MOFA-Flex `0.2.0` API, which is not yet released on PyPI. Install the development version from GitHub:
>
> ```bash
> pip install "mofaflex @ git+https://github.com/bioFAM/mofaflex.git@main"
> ```

In [ ]:
import os
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import torch

import liana as li
import mofaflex as mfl
import squidpy as sq
import decoupler as dc

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import scanpy as sc

from anndata._warnings import ImplicitModificationWarning
warnings.filterwarnings("ignore", category=ImplicitModificationWarning)

# Select the fastest available device for MOFA-Flex training: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

## Load and Prep Data

To demonstrate the integration in practice, we reuse the same spatial transcriptomics dataset as the [inflow score tutorial](https://github.com/saezlab/liana-py/blob/main/docs/notebooks/inflow_score.ipynb): a MERFISH dataset from [*A molecularly defined and spatially resolved cell atlas of the whole mouse brain* (Nature, 2023)](https://www.nature.com/articles/s41586-023-06808-9); [WB_MERFISH_animal2_coronal](https://cellxgene.cziscience.com/collections/0cca8620-8dee-45d0-aef5-23f032a5cf09).

In [ ]:
file_path = "data/MERFISH_mouse_brain/WB_MERFISH_animal2_coronal.h5ad"
backup_url = "https://datasets.cellxgene.cziscience.com/93c3bb97-ea05-4ee0-a760-a1508cd04612.h5ad"

if os.path.exists(file_path):
    adata = sc.read(file_path)
else:
    adata = sc.read(
    file_path,
    backup_url=backup_url
)

In [ ]:
adata = adata[adata.obs["brain_section_label"] == "C57BL6J-2.039"].copy()
adata.var_names = adata.var["gene_name"]

# The spatial coordinates load as float64. MPS (Apple Silicon) does not support float64,
# so we cast them to float32 here. This propagates downstream into the MuData used to train
# MOFA-Flex, letting its Gaussian Process prior (which reads these coordinates) run on MPS.
adata.obsm["X_spatial_coords"] = adata.obsm["X_spatial_coords"].astype("float32")

In [ ]:
sc.pl.embedding(adata, basis="X_spatial_coords", color=["major_brain_region", "cell_type"], wspace=0.4, s=5)

## Basic Prep and QC

In [ ]:
# filter cells and genes
sc.pp.filter_cells(adata, min_genes=10)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## Compute Inflow Score (Condensed Recap)

We briefly recompute the inflow score here, for a full walkthrough of bandwidth selection, spatial connectivity, and spatially variable gene (SVG) filtering, see the [inflow score tutorial](https://github.com/saezlab/liana-py/blob/main/docs/notebooks/inflow_score.ipynb).

Since this dataset is mouse, we go straight to the `mouseconsensus` resource.

In [ ]:
li.ut.spatial_neighbors(adata=adata, bandwidth=30, spatial_key="X_spatial_coords")

sq.gr.spatial_autocorr(adata, mode='moran', use_raw=False, show_progress_bar=True)
svgs = adata.uns['moranI'].index[(adata.uns['moranI']['pval_norm_fdr_bh'] < 0.05) & (adata.uns['moranI']['I'] > 0.01)]
adata = adata[:, svgs]

resource = li.rs.select_resource('mouseconsensus')

lrdata = li.mt.inflow(adata,
                      groupby='cell_type',
                      resource=resource,
                      use_raw=False)
lrdata.shape

### Optional: filter to spatially variable LR interactions

In [ ]:
sq.gr.spatial_autocorr(lrdata, mode='moran', use_raw=False)
svis = lrdata.uns['moranI'].index[(lrdata.uns['moranI']['pval_norm_fdr_bh'] <= 0.05) & (lrdata.uns['moranI']['I'] > 0.01)]
lrdata = lrdata[:, svis]
lrdata.shape

---
## From AnnData to MuData

We need to reshape the inflow score AnnData into a MuData object, where each **sender cell type becomes a view**.

In [ ]:
from liana.multi import lrdata_to_mudata

mdata = lrdata_to_mudata(lrdata, min_features=80, obs_keys=["cell_type", "major_brain_region"])
mdata

---
## Training MOFA-Flex

MOFA-Flex extends the standard MOFA framework with a **Gaussian Process (GP) factor prior**: instead of treating cells as exchangeable, the GP prior encodes spatial proximity, so nearby cells in tissue are expected to have similar factor values. This makes factors spatially interpretable rather than just statistically optimal.

Here we pair the GP factor prior with **Horseshoe weight prior** to promote sparsity in the features loading matrix, so each factor is driven by a small, coherent set of features.

Training can take significant time and memory depending on dataset size and `max_epochs`; for large datasets, consider running this as a background job or on a compute cluster outside the notebook rather than interactively.

In [ ]:
os.makedirs("models", exist_ok=True)

model = mfl.terms.MofaFlex(
    n_factors=10,
    factor_prior=mfl.priors.GaussianProcess(
        covariates_mkey="X_spatial_coords",
        independent_lengthscales=True,
    ),
    weight_prior=mfl.priors.Horseshoe(),
)

model.fit(
    mdata,
    batch_size=2048,
    n_particles=1,
    lr=0.1,
    max_epochs=1000,
    save_path="models/inflow_mofaflex_mouse_brain.hdf5",
    early_stopper_patience=20,
    device=device,
)

---
## Model Quality Checks

Before interpreting factors biologically, we run two diagnostic checks:

1. **Factor correlation**: correlated factor pairs may encode the same signal, indicating the model could be simplified or that a biological program was split across multiple factors.
2. **Variance explained**: quantifies how much of the data variance each factor accounts for, per modality. Factors with near-zero R² are noise and will be removed in the next step.

If you trained the model in a separate process (as suggested above for large datasets), reload it here. If you just ran the training cell above in this notebook, `model` is already in memory and this step is not needed.

In [ ]:
# Optional: reload a previously trained model
model = mfl.MOFAFLEX.load("models/inflow_mofaflex_mouse_brain.hdf5")

### Latent Factor Correlation

Each factor captures an independent biological signal. Highly correlated factors (|r| > 0.6) suggest redundancy.

In [ ]:
# Check for redundant factors: highly correlated factors may capture the same signal
mfl.pl.factor_correlation(model, figsize=(6, 6))

### Variance Explained per Modality

In [ ]:
# How much variance does each factor explain?
# Helps identify which factors are biologically meaningful vs noise
mfl.pl.variance_explained(model, figsize=(6, 6))

---
## Selecting Informative Factors

Not all factors are equally informative. We keep a factor only if it clears a minimum R² floor in at least one modality (the standard definition of an "active"/non-noise factor), then among those, rank by total R² (summed across modalities) and keep the smallest subset that cumulatively explains ≥ 99% of the remaining explained variance.

In [ ]:
r2_df = model.get_r2(type="term", term=None)  # columns: group, view, component, R2

# 1. rank factors that clear a 2% R2 floor in >=1 view/group by their total R2
r2_by_factor = r2_df.groupby("component")["R2"]
ranked = r2_by_factor.sum()[r2_by_factor.max() >= 0.02].sort_values(ascending=False)

# 2. keep the smallest, most informative subset covering 99% of that variance
factors = ranked.index[: (ranked.cumsum() / ranked.sum() < 0.99).sum() + 1].tolist()

# get_factors() returns dict[str, pd.DataFrame] (samples x factors), not AnnData
factor_adata = ad.AnnData(model.get_factors()["group_1"])[:, factors].copy()
factor_adata.obsm["spatial"] = adata.obsm["X_spatial_coords"].copy()

shared_obs = adata.obs_names.intersection(factor_adata.obs_names)
factor_adata.obs.loc[shared_obs, ["cell_type", "major_brain_region"]] = adata.obs.loc[
    shared_obs, ["cell_type", "major_brain_region"]
]

In [ ]:
factor_adata

---
## Exploring Factors: Feature Weights and Spatial Activity

Each MOFA-Flex factor has two complementary representations:

- **Feature weights** (loadings): which LR-pairs drive the factor. Larger absolute weights indicate a stronger association between that feature and the factor.
- **Factor scores**: the per-cell value of the factor. Since the GP prior was used, these scores should vary smoothly across the tissue section.

### Top Feature Weights per Factor

The plot below shows the 5 features (LR pairs) with the largest absolute weight for each factor. Features appearing in multiple factors may represent shared biological themes.

In [ ]:
# For each factor, show the 5 features with the largest absolute weight.
# High-weight features are the molecular signals most strongly associated with that factor.
mfl.pl.top_weights(model, n_features=5, factors=None, figsize=(18, 8), nrow=None, ncol=None)

### Spatial Factor Activity

We now visualise each factor's score directly on the tissue.

In [ ]:
# For better visualization, we use a diverging colormap and set the color limits to be symmetric around zero.
vals = np.hstack([factor_adata[:, f].X.flatten() for f in factors])
# avoid outliers (better for color scaling)
p_low, p_high = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
absmax = max(abs(p_low), abs(p_high))

# make symmetric around 0
norm = colors.TwoSlopeNorm(vcenter=0, vmin=-absmax, vmax=absmax)

sc.pl.embedding(
    factor_adata,
    basis="spatial",
    color=factors,
    cmap="RdBu_r",
    norm=norm,
    s=10,
    ncols=4,
    wspace=0.1,
    show=True
)

---
## Cell Clustering in Factor Space

We build a k-nearest-neighbour (k-NN) graph directly on the factor matrix. This means cells are grouped by their communication program, rather than by raw inflow score similarity.

In [ ]:
# Build a k-NN graph in factor space
sc.pp.neighbors(factor_adata, use_rep='X')

In [ ]:
# Embed the factor-space k-NN graph in 2D for visualization
sc.tl.umap(factor_adata, neighbors_key='neighbors')

In [ ]:
# Cluster cells by communication program (low resolution -> few, coarse clusters)
sc.tl.leiden(factor_adata, resolution=0.4, flavor="igraph")

In [ ]:
# Compare unsupervised Leiden clusters against known annotations
sc.pl.umap(
    factor_adata,
    color=[ "leiden", "major_brain_region", "cell_type"],
    size=8,
)

Overlay each factor's score on the same UMAP to see which factors drive which clusters

In [ ]:
sc.pl.umap(
    factor_adata,
    color=factors,           # factor scores as color
    cmap="RdBu_r",
    vcenter=0,
    size=8,
    ncols=4,
    wspace=0.1,
)

Check whether the Leiden clusters form spatially coherent regions in tissue

In [ ]:
sc.pl.embedding(
    factor_adata,
    basis="spatial",
    color=["leiden", "major_brain_region",  "cell_type"],
    s=8,
    wspace=0.3,
    show=True
)

### Sanity check: is MOFA-Flex factorization capturing something meaningful?

Before interpreting these clusters biologically, it's worth checking whether the factorization step is adding value, or whether the same clusters would emerge from clustering directly on the raw (unfactorized) inflow scores in `lrdata`. We repeat the identical k-NN + Leiden recipe, this time skipping MOFA-Flex entirely, and compare.

In [ ]:
# For comparison: cluster directly on the raw (unfactorized) inflow scores, skipping MOFA-Flex
raw_adata = lrdata.copy()
sc.pp.scale(raw_adata, max_value=10)
sc.tl.pca(raw_adata, n_comps=len(factors))  # match the number of retained MOFA-Flex factors
sc.pp.neighbors(raw_adata, use_rep="X_pca")
sc.tl.leiden(raw_adata, resolution=0.1, flavor="igraph", n_iterations=2)

sc.pl.embedding(
    raw_adata,
    basis="X_spatial_coords",
    color=["leiden", "cell_type"],
    s=8,
    wspace=0.3,
)

Clustering directly on the raw inflow scores produces dozens of small, spatially incoherent clusters, in sharp contrast to the clean, spatially coherent clusters found in factor space above. With ~1,400 noisy `sender^ligand^receptor` features and no spatial prior, PCA has no reason to produce spatially smooth structure. MOFA-Flex's Horseshoe prior enforces sparsity on the loadings and its Gaussian Process prior enforces spatial smoothness on the factors.

Next, we zoom into two of the retained factors, Factor 6 and Factor 10, to see concretely what "a different communication program" looks like in practice.

### Factor 6 vs Factor 10: two distinct communication programs

Both factors passed the R² and correlation filters above, and the factor-correlation diagnostic showed them to be essentially uncorrelated (|r| ≈ 0). That tells us they're statistically independent, but not what biology they actually capture. We compare their top-weighted features and spatial activity side by side to make that concrete.

In [ ]:
mfl.pl.top_weights(model, n_features=5, factors=["Factor 6", "Factor 10"], figsize=(8, 4))

Factor 6 is driven by astrocyte, oligodendrocyte-precursor, and glutamatergic-neuron ligands (`Fgf1`, `Tnc`, `Vcan`, `Fgf13`) converging on `Egfr`, an EGFR-centred growth-factor/ECM signaling axis. Factor 10 instead is dominated by astrocyte- and GABAergic-neuron-derived `Agt` and `Penk` acting on `Adra2a` and the opioid receptors `Oprd1`/`Oprm1`, a neuropeptide, adrenergic/opioid signaling axis. Note that astrocytes send in both factors, but through entirely different ligand-receptor pairs and presumably different receiving cells. The same sender cell type running two unrelated communication programs.

In [ ]:
focus_factors = ["Factor 6", "Factor 10"]
vals = np.hstack([factor_adata[:, f].X.flatten() for f in focus_factors])
p_low, p_high = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
absmax = max(abs(p_low), abs(p_high))
norm = colors.TwoSlopeNorm(vcenter=0, vmin=-absmax, vmax=absmax)

sc.pl.embedding(
    factor_adata,
    basis="spatial",
    color=focus_factors,
    cmap="RdBu_r",
    norm=norm,
    s=10,
    ncols=2,
    wspace=0.15,
    show=True
)

The two factors don't just differ molecularly, they occupy different territory in the tissue. Factor 6 is diffusely active across much of the section, consistent with EGFR signaling occurring wherever astrocytes, OPCs, and glutamatergic neurons happen to sit. Factor 10, by contrast, is concentrated in a small number of sharply-defined spatial hotspots rather than spread throughout the tissue. To pin down exactly which annotated brain regions those hotspots correspond to, we compute each factor's mean score per region.

In [ ]:
region_means = (
    factor_adata.to_df()[focus_factors]
    .groupby(factor_adata.obs["major_brain_region"], observed=True)
    .mean()
    .sort_values("Factor 10", ascending=False)
)

region_means.plot(kind="bar", figsize=(9, 4))
plt.ylabel("Mean factor score")
plt.xlabel("Major brain region")
plt.title("Factor 6 vs Factor 10 activity by brain region")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## Package Versions

For reproducibility, we report the versions of the key packages used to run this tutorial.

In [ ]:
print(f"mofaflex: {mfl.__version__}")
print(f"liana:    {li.__version__}")

---

<div style="border:2px solid #790e0eff; background-color:#e3f2fd; padding:12px; border-radius:8px; font-size:1.1em; margin:10px 0;">
<strong>Note:</strong> This tutorial is part of an ongoing <b>LIANA+</b> extension (Alsayah et al., in prep). Feedback welcome!
</div>